In [ ]:
import urllib.request

def simple_get(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return resp.read()


In [ ]:
import os
import time
import warnings
import requests
import pandas as pd

warnings.filterwarnings('ignore')

# --- Configuration ---
TARGET_YEARS = [2023, 2024, 2025]
TARGET_POCS  = ['OTA2201', 'BEN2201', 'HLY2201']
DATA_DIR     = './data'
USECOLS      = ['TradingDate', 'TradingPeriod', 'PublishDateTime',
                'PointOfConnection', 'Island', 'IsProxyPriceFlag',
                'DollarsPerMegawattHour']

os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://www.emi.ea.govt.nz/Wholesale/Datasets/DispatchAndPricing/DispatchEnergyPrices"

def acquire_emi_data(years, pocs, save_dir, retries=3, sleep=1.0):
    all_data    = []
    failed_urls = []
    today       = pd.Timestamp.now().normalize()

    for year in years:
        print(f"\n{'='*40}\n Year: {year}\n{'='*40}")

        dates = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq="D")

        for day in dates:

            # Skip future dates
            if day > today:
                continue

            ymd        = day.strftime("%Y%m%d")
            filename   = f"{ymd}_DispatchEnergyPrices.csv"
            url        = f"{BASE_URL}/{year}/{filename}"
            local_path = os.path.join(save_dir, filename)

            print(f"  {filename} ...", end=" ", flush=True)

            for attempt in range(1, retries + 1):
                try:
                    r = requests.get(url, stream=True, timeout=120)

                    if r.status_code == 200:
                        # Save raw file temporarily
                        with open(local_path, 'wb') as f:
                            for chunk in r.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)

                        # Read only needed columns, filter to target PoCs
                        df_day      = pd.read_csv(local_path, usecols=USECOLS)
                        df_filtered = df_day[df_day['PointOfConnection'].isin(pocs)].copy()
                        all_data.append(df_filtered)

                       
                        os.remove(local_path)
                        print(f"OK ({len(df_filtered):,} rows)")
                        break

                    elif r.status_code == 404:
                        print("404 - skipped")
                        failed_urls.append(url)
                        break

                    else:
                        print(f"HTTP {r.status_code} (attempt {attempt}/{retries})")

                except requests.RequestException as e:
                    print(f"Error attempt {attempt}/{retries}: {e}")

            time.sleep(sleep)
 
        if all_data:
            checkpoint = pd.concat(all_data, ignore_index=True)
            checkpoint.to_csv(os.path.join(save_dir, 'checkpoint.csv'), index=False)
            print(f"\n  Checkpoint saved ({len(checkpoint):,} rows so far)")

    if not all_data:
        print("\nNo data retrieved. Check URL or network access.")
        return None

    master_df  = pd.concat(all_data, ignore_index=True)
    year_range = f"{min(years)}_{max(years)}"
    final_path = os.path.join(save_dir, f'EMI_Filtered_{year_range}.csv')
    master_df.to_csv(final_path, index=False)

    print(f"\nDone! Saved to       : {final_path}")
    print(f"Total rows           : {len(master_df):,}")
    print(f"Date range           : {master_df['TradingDate'].min()} -> {master_df['TradingDate'].max()}")
    print(f"PoCs found           : {master_df['PointOfConnection'].unique().tolist()}")
    print(f"Failed / missing days: {len(failed_urls)}")

    return master_df


df_raw = acquire_emi_data(TARGET_YEARS, TARGET_POCS, DATA_DIR)

if df_raw is not None:
    display(df_raw.head(10))
    display(df_raw.groupby(['PointOfConnection', 'TradingDate']).size().describe())


 Year: 2023
OK (933 rows)spatchEnergyPrices.csv ... 
  20230102_DispatchEnergyPrices.csv ... OK (924 rows)
  20230103_DispatchEnergyPrices.csv ... OK (894 rows)
  20230104_DispatchEnergyPrices.csv ... OK (882 rows)
  20230105_DispatchEnergyPrices.csv ... OK (792 rows)
  20230106_DispatchEnergyPrices.csv ... OK (783 rows)
  20230107_DispatchEnergyPrices.csv ... OK (801 rows)
  20230108_DispatchEnergyPrices.csv ... OK (894 rows)
  20230109_DispatchEnergyPrices.csv ... OK (924 rows)
  20230110_DispatchEnergyPrices.csv ... OK (957 rows)
  20230111_DispatchEnergyPrices.csv ... OK (918 rows)
  20230112_DispatchEnergyPrices.csv ... OK (918 rows)
  20230113_DispatchEnergyPrices.csv ... OK (909 rows)
  20230114_DispatchEnergyPrices.csv ... OK (750 rows)
  20230115_DispatchEnergyPrices.csv ... OK (894 rows)
  20230116_DispatchEnergyPrices.csv ... OK (951 rows)
  20230117_DispatchEnergyPrices.csv ... OK (900 rows)
  20230118_DispatchEnergyPrices.csv ... OK (900 rows)
  20230119_DispatchEnergyPri

,TradingDate,TradingPeriod,PublishDateTime,PointOfConnection,Island,IsProxyPriceFlag,DollarsPerMegawattHour
0,2023-01-01,1,2022-12-31T23:59:44.000+13:00,HLY2201,NI,N,5.13
1,2023-01-01,1,2022-12-31T23:59:44.000+13:00,OTA2201,NI,N,5.15
2,2023-01-01,1,2022-12-31T23:59:44.000+13:00,BEN2201,SI,N,5.05
3,2023-01-01,1,2023-01-01T00:04:43.000+13:00,HLY2201,NI,N,4.57
4,2023-01-01,1,2023-01-01T00:04:43.000+13:00,OTA2201,NI,N,4.58
5,2023-01-01,1,2023-01-01T00:04:43.000+13:00,BEN2201,SI,N,4.69
6,2023-01-01,1,2023-01-01T00:10:04.000+13:00,HLY2201,NI,N,4.10
7,2023-01-01,1,2023-01-01T00:10:04.000+13:00,OTA2201,NI,N,4.11
8,2023-01-01,1,2023-01-01T00:10:04.000+13:00,BEN2201,SI,N,4.04
9,2023-01-01,1,2023-01-01T00:14:42.000+13:00,HLY2201,NI,N,4.09


count    3219.000000
mean      285.683131
std        14.115047
min       229.000000
25%       278.000000
50%       286.000000
75%       296.000000
max       323.000000
dtype: float64

In [10]:
failed_urls = [] 
for url in failed_urls:
    print(url)

In [2]:
# Loading and Inspect Dataset
import pandas as pd
import os

# Define the path to our saved master dataset
DATA_PATH = './data/EMI_Filtered_2023_2025.csv'

print("Loading dataset into memory...")
df = pd.read_csv(DATA_PATH)

# Convert date columns to proper datetime objects for easier inspection
df['TradingDate'] = pd.to_datetime(df['TradingDate'])
df['PublishDateTime'] = pd.to_datetime(df['PublishDateTime'])

# Basic Shape
print(f"\n1. Dataset Shape")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")

# Check for Missing Values (Crucial for our pipeline)
print("\n2. Missing Values")
print(df.isna().sum())

# Quick look at the Time Range and Target Nodes
print("\n3. Time Range & Nodes")
print(f"Start Date: {df['TradingDate'].min().date()}")
print(f"End Date:   {df['TradingDate'].max().date()}")
print(f"Target PoCs: {df['PointOfConnection'].unique().tolist()}")

# Statistical Summary of the Target Variable (Price Spikes!)
print("\n4. Price Distribution ($/MWh)")
# Formatting to 2 decimal places so it's readable
print(df['DollarsPerMegawattHour'].describe().apply(lambda x: f"{x:,.2f}"))

# Display the first 10 rows to inspect the irregular 'PublishDateTime'
print("\n5. First 10 Rows")
display(df.head(10))

Loading dataset into memory...


/var/folders/x7/vxp2byts2kd3j_zk5_3ksgdm0000gn/T/ipykernel_93944/3460075299.py:13: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['PublishDateTime'] = pd.to_datetime(df['PublishDateTime'])



1. Dataset Shape
Total Rows: 919,614
Total Columns: 7

2. Missing Values
TradingDate               0
TradingPeriod             0
PublishDateTime           0
PointOfConnection         0
Island                    0
IsProxyPriceFlag          0
DollarsPerMegawattHour    0
dtype: int64

3. Time Range & Nodes
Start Date: 2023-01-01
End Date:   2025-12-31
Target PoCs: ['HLY2201', 'OTA2201', 'BEN2201']

4. Price Distribution ($/MWh)
count    919,614.00
mean         157.35
std          154.14
min            0.00
25%           50.00
50%          147.95
75%          230.18
max        9,137.58
Name: DollarsPerMegawattHour, dtype: object

5. First 10 Rows


,TradingDate,TradingPeriod,PublishDateTime,PointOfConnection,Island,IsProxyPriceFlag,DollarsPerMegawattHour
0,2023-01-01,1,2022-12-31 23:59:44+13:00,HLY2201,NI,N,5.13
1,2023-01-01,1,2022-12-31 23:59:44+13:00,OTA2201,NI,N,5.15
2,2023-01-01,1,2022-12-31 23:59:44+13:00,BEN2201,SI,N,5.05
3,2023-01-01,1,2023-01-01 00:04:43+13:00,HLY2201,NI,N,4.57
4,2023-01-01,1,2023-01-01 00:04:43+13:00,OTA2201,NI,N,4.58
5,2023-01-01,1,2023-01-01 00:04:43+13:00,BEN2201,SI,N,4.69
6,2023-01-01,1,2023-01-01 00:10:04+13:00,HLY2201,NI,N,4.10
7,2023-01-01,1,2023-01-01 00:10:04+13:00,OTA2201,NI,N,4.11
8,2023-01-01,1,2023-01-01 00:10:04+13:00,BEN2201,SI,N,4.04
9,2023-01-01,1,2023-01-01 00:14:42+13:00,HLY2201,NI,N,4.09
